# Lab 10 — Fully Connected Network from Scratch
Compare SGD, Momentum, AdaGrad and Adam on healthcare tabular data. Streamlined Colab edition.

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
X,y=load_breast_cancer(return_X_y=True); y=(y==0).astype(int); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)

In [ ]:
def relu(x): return np.maximum(0,x)
def softmax(x): x=x-x.max(1,keepdims=True); e=np.exp(x); return e/e.sum(1,keepdims=True)
class Net:
 def __init__(self,d,h=32):
  r=np.random.default_rng(42); self.p={'W1':r.normal(0,np.sqrt(2/d),(d,h)),'b1':np.zeros(h),'W2':r.normal(0,np.sqrt(2/h),(h,2)),'b2':np.zeros(2)}
 def loss_grad(self,X,y):
  z1=X@self.p['W1']+self.p['b1']; a=relu(z1); p=softmax(a@self.p['W2']+self.p['b2']); n=len(y); loss=-np.log(p[np.arange(n),y]+1e-9).mean(); dz=p; dz[np.arange(n),y]-=1; dz/=n; g={'W2':a.T@dz,'b2':dz.sum(0)}; da=dz@self.p['W2'].T; dz1=da*(z1>0); g['W1']=X.T@dz1; g['b1']=dz1.sum(0); return loss,g
 def predict(self,X): return softmax(relu(X@self.p['W1']+self.p['b1'])@self.p['W2']+self.p['b2']).argmax(1)

In [ ]:
def train(kind,epochs=120,lr=.01):
 net=Net(Xtr.shape[1]); state={k:np.zeros_like(v) for k,v in net.p.items()}; state2={k:np.zeros_like(v) for k,v in net.p.items()}; losses=[]
 for t in range(1,epochs+1):
  loss,g=net.loss_grad(Xtr,ytr); losses.append(loss)
  for k in net.p:
   if kind=='SGD': net.p[k]-=lr*g[k]
   elif kind=='Momentum': state[k]=.9*state[k]-lr*g[k]; net.p[k]+=state[k]
   elif kind=='AdaGrad': state[k]+=g[k]**2; net.p[k]-=lr*g[k]/(np.sqrt(state[k])+1e-8)
   else: state[k]=.9*state[k]+.1*g[k]; state2[k]=.999*state2[k]+.001*g[k]**2; mh=state[k]/(1-.9**t); vh=state2[k]/(1-.999**t); net.p[k]-=lr*mh/(np.sqrt(vh)+1e-8)
 return net,losses
results={}
for k in ['SGD','Momentum','AdaGrad','Adam']:
 net,losses=train(k); results[k]=(accuracy_score(yte,net.predict(Xte)),losses); print(k,results[k][0])

In [ ]:
for k,(acc,l) in results.items(): plt.plot(l,label=k)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()